In [ ]:
# imports

import os
import requests
from dataclasses import dataclass, field
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

In [ ]:
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
OPEN_AI_MODEL = "gpt-4.1-mini"
ANTHROPIC_MODEL = "claude-sonnet-4-5-20250929"
GEMINI_MODEL = "gemini-2.5-pro"

In [ ]:
# Same Alex / Blake / Charlie prompts are used twice at the bottom:
# 1) Paid models: OpenAI + Claude + Gemini
# 2) Ollama as 3 separate users

# Going local

Just use the OpenAI library pointed to localhost:11434/v1

### If not running, run ollama serve at a command line

### If you do not have ollama on machine run "!ollama pull llama3.2"

### Only do this if you have a large machine - at least 16GB RAM run "ollama pull gpt-oss:20b"

In [ ]:
OLLAMA_MODEL = "llama3.2"

In [ ]:
# GLOBALS
ollama_api_key = "ollama_api_key"
ollama_url = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1/')
ollama = OpenAI(api_key=ollama_api_key, base_url=ollama_url)

In [ ]:
# Needed only for the Ollama run. Paid-model run still works if Ollama is down.
try:
    response = requests.get("http://localhost:11434/", timeout=2).content
    if b"Ollama" in response:
        display(Markdown("✅ **Ollama server is running.**"))
    else:
        display(Markdown("⚠️ Ollama responded, but the body did not look like the usual health check."))
except Exception:
    display(Markdown("⚠️ **Ollama is not running.** You can still run the paid-model debate. Start `ollama serve` before the local 3-user run."))


# System Prompt Design for Chatbot interaction

In [ ]:
## This is GPT-4o system prompt and Same can be used for fake user for ollama
Alex_system_prompt = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation
and you challenge everything with ferver. You are the the debate moderator engaged in a debate discussion
where Blake and Charlie have opposing views about technology.
You only ask questions and make comments to keep the debate lively. You do not take sides.
You do not engage in the debate yourself. When you are called by the API, 
"""

## This is CLAUDE who is a fake user and Blake system prompt and Same can be used for fake user for ollama
Blake_system_prompt = """
You are Blake, a chatbot who is very clever and a bit snarky. 
You are clever and funny. 
You are engaged in a debate discussion with Charlie about technology.
Alex is the debate moderator and you need to respond to his questions and comments.
When you are called by the API, you only respond to the moderator's question.
"""
## This is GEMINI who is a fake user and Blake system prompt and Same can be used for fake user for ollama
Charlie_system_prompt = """
You are Charlie, a chatbot who is very polite, courteous and a bit argumentative. 
You always try to find common ground with the opposing side. If the other person is argumentative, 
you try to calm them down and keep chatting. If they say something funny, you laugh politely. 
You are engaged in a debate discussion with Blake about technology.
"""

Alex_messages = ["Hello, I am alex ! I am the debate moderator In this debate I will be asking questions to both sides and keep the debate going. The topic of discuss is technology"]
Blake_messages = ["That's quite the topic. "]
Charlie_messages = ["Hello!, I am Charlie. Pleasure to meet you both. technology is such an interesting topic!"]


# Using Paid model from p

In [ ]:
def alex_call():
    messages = [{"role": "system", "content": Alex_system_prompt}]

    for alex, blake, charlie in zip(Alex_messages, Blake_messages, Charlie_messages):
        messages.append({"role":"assistant", "content":alex})
        messages.append({"role":"user", "content":blake})
        messages.append({"role":"user", "content":charlie})

    response = openai.chat.completions.create(model=OPEN_AI_MODEL, messages=messages, max_tokens=500)
    return response.choices[0].message.content

def blake_call():
    messages = [{"role": "system", "content": Blake_system_prompt}]

    for alex, blake, charlie in zip(Alex_messages, Blake_messages, Charlie_messages):
        messages.append({"role":"assistant", "content":blake})
        messages.append({"role":"user", "content":alex})
        messages.append({"role":"user", "content":charlie})

    response = openai.chat.completions.create(model=OPEN_AI_MODEL, messages=messages, max_tokens=500)
    return response.choices[0].message.content

def charlie_call():
    messages = [{"role": "system", "content": Charlie_system_prompt}]

    for alex, blake, charlie in zip(Alex_messages, Blake_messages, Charlie_messages):
        messages.append({"role":"assistant", "content":charlie})
        messages.append({"role":"user", "content":alex})
        messages.append({"role":"user", "content":blake})

    response = openai.chat.completions.create(model=OPEN_AI_MODEL, messages=messages, max_tokens=500)
    return response.choices[0].message.content


In [ ]:
display(Markdown(f'### Alex :\n{Alex_messages}'))
display(Markdown(f'### Blake :\n{Blake_messages}'))
display(Markdown(f'### Charlie :\n{Charlie_messages}'))

for i in range(10):
    alex_response = alex_call()
    display(Markdown(f'### Alex :\n{alex_response}'))
    blake_response = blake_call()
    display(Markdown(f'### Blake :\n{blake_response}'))
    charlie_response = charlie_call()
    display(Markdown(f'### Charlie :\n{charlie_response}'))
    print("\n")


# Generic debate functions

Your original `alex_call` / `blake_call` / `charlie_call` cells above are unchanged.

These helpers reuse the **same** Alex, Blake, and Charlie system prompts:
1. **Paid models** — Alex=OpenAI, Blake=Claude, Charlie=Gemini
2. **Ollama as 3 users** — the same three people, all served by Ollama


In [ ]:
@dataclass
class Participant:
    """One speaker: persona (name + system prompt) plus whichever client/model plays them."""
    name: str
    client: OpenAI
    model: str
    system_prompt: str
    messages: list = field(default_factory=list)


def build_messages(speaker: Participant, participants: list) -> list:
    """Each speaker sees their own lines as assistant and everyone else as user."""
    history = [{"role": "system", "content": speaker.system_prompt}]
    n_rounds = max(len(p.messages) for p in participants)
    for i in range(n_rounds):
        for person in participants:
            if i >= len(person.messages):
                continue
            role = "assistant" if person is speaker else "user"
            history.append({"role": role, "content": f"{person.name}: {person.messages[i]}"})
    return history


def call_participant(speaker: Participant, participants: list, max_tokens: int = 500) -> str:
    messages = build_messages(speaker, participants)
    response = speaker.client.chat.completions.create(
        model=speaker.model,
        messages=messages,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


def debate_users(alex, blake, charlie):
    """Same Alex / Blake / Charlie prompts. Pass (client, model) for each user."""
    return [
        Participant("Alex", alex[0], alex[1], Alex_system_prompt, list(Alex_messages)),
        Participant("Blake", blake[0], blake[1], Blake_system_prompt, list(Blake_messages)),
        Participant("Charlie", charlie[0], charlie[1], Charlie_system_prompt, list(Charlie_messages)),
    ]


def run_conversation(participants: list, rounds: int = 5):
    for person in participants:
        display(Markdown(f"### {person.name} (`{person.model}`):\n{person.messages[-1]}"))

    for _ in range(rounds):
        for speaker in participants:
            reply = call_participant(speaker, participants)
            speaker.messages.append(reply)
            display(Markdown(f"### {speaker.name} (`{speaker.model}`):\n{reply}"))


In [ ]:
# Run 1 — paid models, same Alex / Blake / Charlie prompts
paid_debate = debate_users(
    alex=(openai, OPEN_AI_MODEL),
    blake=(anthropic, ANTHROPIC_MODEL),
    charlie=(gemini, GEMINI_MODEL),
)
run_conversation(paid_debate, rounds=5)


## Run 2 — Ollama as 3 separate users

Same Alex, Blake, and Charlie system prompts. Ollama plays each user in turn.


In [ ]:
# Run 2 — Ollama as 3 separate users (same prompts as Run 1)
ollama_debate = debate_users(
    alex=(ollama, OLLAMA_MODEL),
    blake=(ollama, OLLAMA_MODEL),
    charlie=(ollama, OLLAMA_MODEL),
)
run_conversation(ollama_debate, rounds=5)
